In [ ]:
# ============================================================================
# 🚀 AGGRESSIVE GPU MEMORY RESET (Run First!)
# ============================================================================

import torch
import gc
import os
import subprocess

print("🧹 AGGRESSIVE GPU MEMORY RESET...")

# ============================================================================
# 1. Clear PyTorch cache
# ============================================================================

torch.cuda.empty_cache()
gc.collect()

# ============================================================================
# 2. Reset CUDA memory stats
# ============================================================================

if torch.cuda.is_available():
    try:
        # Reset peak memory stats
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        
        # Clear memory on all GPUs
        for i in range(torch.cuda.device_count()):
            with torch.cuda.device(i):
                torch.cuda.empty_cache()
                gc.collect()
                print(f"✅ GPU {i} cache cleared")
                
    except Exception as e:
        print(f"⚠️ GPU reset warning: {e}")

# ============================================================================
# 3. Force CUDA context reset (nuclear option)
# ============================================================================

try:
    # This forces a full CUDA context reset
    if torch.cuda.is_available():
        # Get current device
        current_device = torch.cuda.current_device()
        
        # Reset the device
        torch.cuda.reset_accumulated_memory_stats()
        
        # Force a synchronization
        torch.cuda.synchronize()
        
        print("✅ CUDA context reset")
except Exception as e:
    print(f"⚠️ CUDA context reset warning: {e}")

# ============================================================================
# 4. Check memory after reset
# ============================================================================

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        allocated = torch.cuda.memory_allocated(i) / 1e9
        reserved = torch.cuda.memory_reserved(i) / 1e9
        print(f"✅ GPU {i}: {allocated:.2f} GB allocated, {reserved:.2f} GB reserved")

print("=" * 70)
print("✅ GPU memory reset complete!")
print("=" * 70)

In [ ]:
import torch
print(f"✅ GPU Available: {torch.cuda.is_available()}")
print(f"✅ GPU Name: {torch.cuda.get_device_name(0)}")
print(f"✅ GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
!pip install -q fastapi uvicorn pyngrok transformers peft accelerate bitsandbytes sentence-transformers faiss-gpu pillow requests python-multipart

In [ ]:
from pyngrok import ngrok
ngrok.kill()
print("✅ ngrok killed")

In [ ]:
!ngrok authtoken 3H5AjFP4WxLLRslJ1fdT9W0CrwZ_5RBJsB6pTTxzQbRKGtK2j

In [ ]:
# # ============================================================================
# # SMART NGROK HANDLER - SKIP IN BACKGROUND MODE
# # ============================================================================

# import os
# import time
# import threading
# import nest_asyncio

# nest_asyncio.apply()

# # ============================================================================
# # DETECT RUN TYPE
# # ============================================================================

# run_type = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', 'Interactive')
# is_background = (run_type == 'Batch')

# print(f"🔍 Detected run type: {run_type}")
# print(f"ℹ️  Is background run: {is_background}")

# os.environ['IS_KAGGLE_BACKGROUND'] = str(is_background)

# # ============================================================================
# # SETUP NGROK (ONLY IN INTERACTIVE SESSIONS)
# # ============================================================================

# if is_background:
#     print("🚀 Background mode (Save Version). Skipping ngrok entirely.")
#     print("📝 For background runs, use the Kaggle logs to check server status.")
#     print("   The API will start without a public URL.")
#     ngrok_url = "N/A - Background Run"
#     os.environ['KAGGLE_API_URL'] = ngrok_url
    
# else:
#     print("🚀 Interactive mode. Setting up ngrok...")
    
#     from pyngrok import ngrok
    
#     # Kill old ngrok
#     try:
#         ngrok.kill()
#     except:
#         pass
    
#     # Start ngrok
#     public_url = ngrok.connect(8080)
#     print(f"✅ ngrok tunnel established!")
#     print(f"🔗 Public URL: {public_url}")
    
#     ngrok_url = public_url.public_url
#     print(f"📝 Your Kaggle API URL: {ngrok_url}")
    
#     os.environ['KAGGLE_API_URL'] = ngrok_url

# print("=" * 70)
# print("✅ Ngrok handler complete!")
# if not is_background:
#     print(f"📝 URL: {ngrok_url}")
# print("=" * 70)

# # ============================================================================
# # START THE SERVER (ADD THIS AFTER NGROK)
# # ============================================================================

# import uvicorn
# import threading

# def start_server():
#     print("🚀 Starting FastAPI server...")
#     uvicorn.run(app, host="0.0.0.0", port=8080, log_level="info")

# # Start server in background thread
# server_thread = threading.Thread(target=start_server, daemon=True)
# server_thread.start()
# time.sleep(2)

# print(f"✅ Server running at: {ngrok_url}")
# print(f"📖 Health check: {ngrok_url}/health")

In [ ]:
import os
import json
import time
import torch
import gc
import numpy as np
from fastapi import FastAPI, UploadFile, File, Request
from fastapi.responses import JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
import requests
from io import BytesIO
import base64
from pyngrok import ngrok
import uvicorn
import threading
import asyncio
import nest_asyncio
from typing import Dict, Any, Optional, List

# Apply nest_asyncio to allow nested event loops
nest_asyncio.apply()

print("✅ All imports successful!")
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
print(f"✅ GPU Count: {torch.cuda.device_count()}")

In [ ]:
class ModelManager:
    """Manages GPU models with lazy loading and memory cleanup"""
    
    def __init__(self):
        self._qlora_model = None
        self._qlora_tokenizer = None
        self._embedding_model = None
        self._vision_model = None
        self._vision_processor = None
        self.current_model = None
        self.last_used = 0
        self.model_ttl = 300  # 5 minutes idle timeout
        
    def unload_all(self):
        """Free all GPU memory"""
        print("🧹 Unloading all models...")
        
        if self._qlora_model is not None:
            del self._qlora_model
            self._qlora_model = None
        if self._qlora_tokenizer is not None:
            del self._qlora_tokenizer
            self._qlora_tokenizer = None
        if self._embedding_model is not None:
            del self._embedding_model
            self._embedding_model = None
        if self._vision_model is not None:
            del self._vision_model
            self._vision_model = None
        if self._vision_processor is not None:
            del self._vision_processor
            self._vision_processor = None
            
        gc.collect()
        torch.cuda.empty_cache()
        self.current_model = None
        print("✅ GPU memory cleared!")
        
    def get_qlora(self):
        """Lazy load QLoRA model"""
        if self._qlora_model is None:
            print("📥 Loading QLoRA model...")
            # We'll fill this in the next step
            pass
        return self._qlora_model, self._qlora_tokenizer
    
    def get_embedding_model(self):
        """Lazy load Sentence Transformer"""
        if self._embedding_model is None:
            print("📥 Loading Sentence Transformer...")
            # We'll fill this in the next step
            pass
        return self._embedding_model
    
    def get_vision_model(self):
        """Lazy load Vision model"""
        if self._vision_model is None:
            print("📥 Loading Vision model...")
            # We'll fill this in the next step
            pass
        return self._vision_model, self._vision_processor

# Create global instance
model_manager = ModelManager()
print("✅ Model Manager created!")

In [ ]:
from sentence_transformers import SentenceTransformer

print("📥 Loading Sentence Transformer...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embedding_model = embedding_model.to('cuda')
print("✅ Sentence Transformer loaded on GPU!")

# Test the embedding model
test_text = "Insurance policy covers car damage"
test_embedding = embedding_model.encode(test_text)
print(f"✅ Test embedding shape: {test_embedding.shape}")
print(f"✅ Test embedding sample: {test_embedding[:5]}")

# Store in model manager
model_manager._embedding_model = embedding_model
print("✅ Embedding model stored in Model Manager!")

In [ ]:
# Create FastAPI app
app = FastAPI(title="Kaggle GPU API", version="1.0.0")

# Add CORS middleware
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
async def root():
    return {
        "status": "online",
        "service": "Kaggle GPU API",
        "models": {
            "embedding": "loaded" if model_manager._embedding_model is not None else "not loaded",
            "qlora": "loaded" if model_manager._qlora_model is not None else "not loaded",
            "vision": "loaded" if model_manager._vision_model is not None else "not loaded"
        },
        "gpu_info": {
            "available": torch.cuda.is_available(),
            "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None",
            "memory_total_gb": torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
        }
    }

@app.get("/health")
async def health():
    return {"status": "healthy", "timestamp": time.time()}

print("✅ FastAPI app created!")
print("📋 Endpoints ready: / (info) and /health")

In [ ]:
from pydantic import BaseModel

class EmbedRequest(BaseModel):
    text: str
    normalize: bool = True

@app.post("/api/embed")
async def embed_text(request: EmbedRequest):
    """Generate embeddings for a single text"""
    try:
        model = model_manager._embedding_model
        if model is None:
            return JSONResponse(
                status_code=503,
                content={"error": "Embedding model not loaded"}
            )
        
        embedding = model.encode(request.text, normalize_embeddings=request.normalize)
        
        return {
            "success": True,
            "embedding": embedding.tolist(),
            "shape": len(embedding),
            "text": request.text[:100] + "..." if len(request.text) > 100 else request.text
        }
    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={"error": str(e)}
        )

@app.post("/api/embed/batch")
async def embed_batch(request: dict):
    """Generate embeddings for multiple texts"""
    try:
        texts = request.get("texts", [])
        if not texts:
            return JSONResponse(
                status_code=400,
                content={"error": "No texts provided"}
            )
        
        model = model_manager._embedding_model
        if model is None:
            return JSONResponse(
                status_code=503,
                content={"error": "Embedding model not loaded"}
            )
        
        embeddings = model.encode(texts, normalize_embeddings=True)
        
        return {
            "success": True,
            "embeddings": embeddings.tolist(),
            "count": len(texts),
            "shape": embeddings.shape[1] if len(embeddings.shape) > 1 else 384
        }
    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={"error": str(e)}
        )

print("✅ Embedding endpoints added:")
print("   - POST /api/embed      (single text)")
print("   - POST /api/embed/batch (multiple texts)")

In [ ]:
# import nest_asyncio
# import asyncio
# import threading
# import time
# from pyngrok import ngrok

# nest_asyncio.apply()

# # Kill any existing ngrok tunnels
# try:
#     ngrok.kill()
# except:
#     pass

# # Start ngrok tunnel on port 8080
# public_url = ngrok.connect(8080)
# print(f"✅ ngrok tunnel established!")
# print(f"🔗 Public URL: {public_url}")

# # Get the actual URL string
# ngrok_url = public_url.public_url
# print(f"📝 Your Kaggle API URL: {ngrok_url}")

# # Store for later use
# import os
# os.environ['KAGGLE_API_URL'] = ngrok_url

# # ============ FIXED SERVER STARTUP ============
# import uvicorn
# import asyncio

# def run_server():
#     """Start the FastAPI server with the correct configuration"""
#     try:
#         # Use the correct uvicorn run method
#         uvicorn.run(
#             app,
#             host="0.0.0.0",
#             port=8080,
#             log_level="info",
#             loop="asyncio"  # ✅ Explicitly set the loop
#         )
#     except TypeError as e:
#         print(f"⚠️ TypeError: {e}")
#         # Fallback: use the simpler method
#         print("🔄 Trying fallback method...")
#         uvicorn.run(app, host="0.0.0.0", port=8080)

# # Start server in a background thread
# server_thread = threading.Thread(target=run_server, daemon=True)
# server_thread.start()
# time.sleep(3)  # Wait for server to start

# print("\n✅ Server is running!")
# print(f"📍 API URL: {ngrok_url}")
# print(f"📖 Health check: {ngrok_url}/health")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import torch

print("📥 Loading QLoRA model (Qwen2.5-7B with 4-bit quantization)...")

# ============================================================================
# Step 1: Configure 4-bit quantization
# ============================================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("✅ 4-bit quantization configured!")

# ============================================================================
# Step 2: Load base model with 4-bit quantization
# ============================================================================

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    torch_dtype=torch.float16,
    device_map="auto",
    quantization_config=bnb_config,  # ✅ Pass the config here
    trust_remote_code=True,
)

print("✅ Base model loaded with 4-bit quantization!")

# ============================================================================
# Step 3: Load tokenizer
# ============================================================================

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("✅ Tokenizer loaded!")

# ============================================================================
# Step 4: For now, skip adapter loading (we'll add this later)
# ============================================================================

qlora_model = base_model

print("✅ QLoRA model ready!")
print(f"📊 Model device: {next(qlora_model.parameters()).device}")
print(f"📊 Model dtype: {next(qlora_model.parameters()).dtype}")

# ============================================================================
# Step 5: Store in model manager
# ============================================================================

# Create model_manager if it doesn't exist
if 'model_manager' not in globals():
    class ModelManager:
        pass
    model_manager = ModelManager()

model_manager._qlora_model = qlora_model
model_manager._qlora_tokenizer = tokenizer
print("✅ QLoRA model stored in Model Manager!")

# ============================================================================
# Step 6: Test generation
# ============================================================================

print("\n🔄 Testing generation...")

prompt = "Write a Python function to calculate the nth Fibonacci number"
inputs = tokenizer(prompt, return_tensors="pt")
inputs = {k: v.to(base_model.device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = base_model.generate(
        inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.1,
        use_cache=True,
    )

generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\n" + "=" * 70)
print("🤖 GENERATED RESPONSE:")
print("=" * 70)
print(generated_text)

# ============================================================================
# Step 7: Show memory usage
# ============================================================================

if torch.cuda.is_available():
    print("\n" + "=" * 70)
    print("📊 GPU MEMORY USAGE:")
    print("=" * 70)
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    
    print(f"💾 Allocated: {allocated:.2f} GB")
    print(f"💾 Reserved: {reserved:.2f} GB")
    print(f"💾 Total GPU memory: {total:.2f} GB")
    print(f"💾 Free GPU memory: {total - allocated:.2f} GB")

print("\n✅ Done!")

In [ ]:
!pip install -q transformers bitsandbytes accelerate

In [ ]:
# Test with different prompts
# prompts = [
#     "Explain quantum computing in simple terms",
#     "Write a Python function to sort a list using quicksort",
#     "What is the capital of France and its population?",
#     "Write a short poem about artificial intelligence",
#     "Explain how backpropagation works in neural networks"
# ]
# 
# for prompt in prompts:
#     print(f"\n{'='*70}")
#     print(f"📝 Prompt: {prompt}")
#     print('='*70)
#     
#     inputs = tokenizer(prompt, return_tensors="pt")
#     inputs = {k: v.to(base_model.device) for k, v in inputs.items()}
#     
#     with torch.no_grad():
#         outputs = base_model.generate(
#             inputs['input_ids'],
#             attention_mask=inputs['attention_mask'],
#             max_new_tokens=200,
#             do_sample=True,
#             temperature=0.7,
#             top_p=0.9,
#             pad_token_id=tokenizer.eos_token_id,
#             repetition_penalty=1.1,
#         )
#     
#     response = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     print(f"🤖 Response:\n{response}")

In [ ]:
import faiss
import json
import numpy as np
from pathlib import Path

# ============================================================================
# Load and index your insurance data for RAG
# ============================================================================

# Load insurance data
def load_insurance_data(file_path="insurance_qa_large.jsonl"):
    """Load insurance QA data from JSONL file"""
    data = []
    if Path(file_path).exists():
        with open(file_path, 'r') as f:
            for line in f:
                try:
                    data.append(json.loads(line))
                except:
                    continue
    return data

insurance_data = load_insurance_data()
print(f"✅ Loaded {len(insurance_data)} insurance QA pairs")

if len(insurance_data) > 0:
    # Create FAISS index
    texts = [item['instruction'] for item in insurance_data]
    embeddings = embedding_model.encode(texts, show_progress_bar=True)
    
    # Normalize embeddings
    embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    
    # Create FAISS index
    dimension = embeddings.shape[1]
    faiss_index = faiss.IndexFlatIP(dimension)  # Inner product for cosine similarity
    faiss_index.add(embeddings.astype('float32'))
    
    print(f"✅ FAISS index created with {faiss_index.ntotal} vectors")
else:
    # Sample data if no file found
    sample_data = [
        {"instruction": "How much for a 38-year-old with $50,000 car?", "response": "Your premium is $1500.00/month or $18000.00/year. At 38 years old with a $50,000 vehicle, the age factor is 0.8 and vehicle factor is 2.50."},
        {"instruction": "What's my monthly premium for a 27-year-old with a $25,000 car?", "response": "Your monthly premium is $750.00. Breakdown: Base $500 × Age factor 0.8 × Vehicle factor 1.25 × Coverage factor 1.5 = $750.00."},
        {"instruction": "Does auto insurance cover liability?", "response": "Liability is covered under your auto policy."},
        {"instruction": "Can I claim for flood damage?", "response": "Yes, flood damage is typically covered under your auto policy. File a claim within 24 hours."},
    ]
    insurance_data = sample_data
    texts = [item['instruction'] for item in insurance_data]
    embeddings = embedding_model.encode(texts, show_progress_bar=True)
    embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
    dimension = embeddings.shape[1]
    faiss_index = faiss.IndexFlatIP(dimension)
    faiss_index.add(embeddings.astype('float32'))
    print(f"⚠️ Using sample data. FAISS index created with {faiss_index.ntotal} vectors")

# Store in model manager
model_manager._insurance_data = insurance_data
model_manager._faiss_index = faiss_index
print("✅ RAG data and FAISS index stored!")

# ============================================================================
# RAG endpoint
# ============================================================================

class RAGRequest(BaseModel):
    query: str
    top_k: int = 3
    use_rag: bool = True

@app.post("/api/rag/query")
async def rag_query(request: RAGRequest):
    """Query with RAG retrieval"""
    try:
        # Get embedding model
        embed_model = model_manager._embedding_model
        if embed_model is None:
            return JSONResponse(503, {"error": "Embedding model not loaded"})
        
        # Check if FAISS index exists
        if not hasattr(model_manager, '_faiss_index'):
            return JSONResponse(503, {"error": "RAG index not initialized"})
        
        # Encode query
        query_embedding = embed_model.encode([request.query], normalize_embeddings=True)
        
        # Search
        distances, indices = model_manager._faiss_index.search(query_embedding.astype('float32'), request.top_k)
        
        # Get results
        results = []
        for idx, dist in zip(indices[0], distances[0]):
            if idx < len(model_manager._insurance_data):
                results.append({
                    "instruction": model_manager._insurance_data[idx]['instruction'],
                    "response": model_manager._insurance_data[idx]['response'],
                    "similarity": float(dist)
                })
        
        # Build context
        context = "Based on the following insurance information:\n\n"
        for i, result in enumerate(results, 1):
            context += f"{i}. Question: {result['instruction']}\n"
            context += f"   Answer: {result['response']}\n\n"
        
        # Generate response with QLoRA
        if request.use_rag and model_manager._qlora_model is not None:
            qlora_model = model_manager._qlora_model
            tokenizer = model_manager._qlora_tokenizer
            
            prompt = f"""<|im_start|>system
You are an insurance assistant. Use the following context to answer questions accurately.
{context}
<|im_end|>
<|im_start|>user
{request.query}
<|im_end|>
<|im_start|>assistant
"""
            
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
            inputs = {k: v.to(qlora_model.device) for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = qlora_model.generate(
                    inputs['input_ids'],
                    max_new_tokens=200,
                    do_sample=True,
                    temperature=0.7,
                    top_p=0.9,
                    pad_token_id=tokenizer.eos_token_id,
                )
            
            response = tokenizer.decode(outputs[0], skip_special_tokens=True)
            # Extract assistant response
            if "assistant\n" in response:
                response = response.split("assistant\n")[-1].strip()
        else:
            # Without RAG, just use top result
            response = results[0]['response'] if results else "No answer found"
        
        return {
            "success": True,
            "query": request.query,
            "context": context if request.use_rag else None,
            "response": response,
            "sources": results,
            "top_k": request.top_k
        }
        
    except Exception as e:
        return JSONResponse(500, {"error": str(e)})

print("✅ RAG endpoints added!")
print("   - POST /api/rag/query")

In [ ]:
from peft import PeftModel
import torch

print("=" * 70)
print("📥 LOADING FINE-TUNED QLORA ADAPTER")
print("=" * 70)

# ✅ CORRECT PATH - Kaggle dataset
adapter_path = "/kaggle/input/datasets/bhishmakhettri/qwen-insurance-adapter"

# Check if adapter exists
import os
if os.path.exists(adapter_path):
    print(f"✅ Adapter found at: {adapter_path}")
else:
    print(f"❌ Adapter not found at: {adapter_path}")
    print("Please make sure the adapter dataset is added as input")
    raise FileNotFoundError("Adapter not found")

# Load the adapter onto the base model
print("🔄 Loading adapter...")
qlora_model = PeftModel.from_pretrained(base_model, adapter_path)
qlora_model.eval()
print("✅ Fine-tuned QLoRA model ready!")

# Store in model manager
model_manager._qlora_model = qlora_model
model_manager._qlora_tokenizer = tokenizer

print(f"📊 Model device: {next(qlora_model.parameters()).device}")
print(f"📊 Model dtype: {next(qlora_model.parameters()).dtype}")

# Test the loaded model
print("\n🧪 Testing loaded model...")
test_query = "Does auto insurance cover flood damage?"
prompt = f"<|im_start|>user\n{test_query}<|im_end|>\n<|im_start|>assistant\n"

inputs = tokenizer(prompt, return_tensors="pt").to('cuda')
with torch.no_grad():
    outputs = qlora_model.generate(
        inputs['input_ids'],
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
if "assistant\n" in response:
    response = response.split("assistant\n")[-1].strip()

print(f"✅ Test successful!")
print(f"📝 Query: {test_query}")
print(f"🤖 Response: {response[:200]}...")

print("\n✅ QLoRA adapter loaded and ready!")

In [ ]:
@app.post("/api/qlora/query")
async def qlora_query(request: dict):
    """Query the fine-tuned QLoRA model"""
    query = request.get("query", "")
    
    if not query:
        return JSONResponse(400, {"error": "Query required"})
    
    try:
        # Format prompt for insurance questions
        prompt = f"<|im_start|>user\n{query}<|im_end|>\n<|im_start|>assistant\n"
        
        inputs = tokenizer(prompt, return_tensors="pt").to('cuda')
        
        with torch.no_grad():
            outputs = qlora_model.generate(
                inputs['input_ids'],
                max_new_tokens=300,
                temperature=0.7,
                do_sample=True,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        # Clean up response
        if "assistant\n" in response:
            response = response.split("assistant\n")[-1].strip()
        if "user\n" in response:
            response = response.split("user\n")[0].strip()
        
        return {
            "success": True,
            "query": query,
            "response": response,
            "model": "qwen-7b-insurance-finetuned"
        }
        
    except Exception as e:
        return JSONResponse(500, {"error": str(e)})

print("✅ QLoRA endpoint added: POST /api/qlora/query")

In [ ]:
# ============================================================================
# INSTALL REQUIRED PACKAGES
# ============================================================================

print("📥 Installing required packages...")
!pip install -q transformers accelerate torch pillow
print("✅ Packages installed!")

# ============================================================================
# IMPORTS
# ============================================================================

import torch
from PIL import Image
import io
import base64
import json
import os
import sys
import traceback
from fastapi import UploadFile, File
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor  # ✅ FIXED
import requests
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("=" * 70)
print("📥 LOADING QWEN2.5-VL-3B FROM HUGGING FACE")
print("=" * 70)

# ✅ Device detection
vision_device = torch.device("cuda:1" if torch.cuda.is_available() and torch.cuda.device_count() > 1 else "cuda")
print(f"✅ Vision model will use: {vision_device}")

# ============================================================================
# LOAD MODEL WITH CORRECT CLASS
# ============================================================================

print("📥 Loading Qwen2.5-VL-3B from Hugging Face...")
model_name = "Qwen/Qwen2.5-VL-3B-Instruct"

try:
    # ✅ Use the correct model class
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map=str(vision_device) if vision_device.type == "cuda" else "auto",
        trust_remote_code=True,
    )
    processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
    print("✅ Qwen2.5-VL-3B loaded successfully!")
except Exception as e:
    logger.error(f"Failed to load: {e}")
    raise

# ============================================================================
# VISION PROCESSOR
# ============================================================================

class VisionProcessor:
    def __init__(self):
        self.model = model
        self.processor = processor
        self.device = vision_device
        self.available = model is not None
        print(f"✅ VisionProcessor initialized on {self.device}")
    
    def process_image(self, file: UploadFile) -> Image.Image:
        contents = file.file.read()
        image = Image.open(io.BytesIO(contents))
        
        if image.mode in ('RGBA', 'LA', 'P'):
            rgb_img = Image.new('RGB', image.size, (255, 255, 255))
            rgb_img.paste(image, mask=image.split()[-1] if image.mode == 'RGBA' else None)
            image = rgb_img
        elif image.mode != 'RGB':
            image = image.convert('RGB')
        
        return image
    
    def identify_image_type(self, image: Image.Image):
        try:
            prompt = """You are an insurance AI classifier. Determine what this image shows.

Answer with ONE word:
- "CAR_DAMAGE" if image shows a vehicle with visible damage (dent, scratch, crack, broken parts, collision damage)
- "INJURY" if image shows a person with visible injury (cut, bruise, burn, wound, bleeding, broken bone, bandage, medical situation)
- "OTHER" for anything else

Answer:"""
            
            conversation = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image},
                        {"type": "text", "text": prompt}
                    ]
                }
            ]
            
            text_prompt = processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
            inputs = processor(
                text=[text_prompt],
                images=[image],
                padding=True,
                return_tensors="pt"
            )
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=20,
                    do_sample=False,
                    pad_token_id=processor.tokenizer.eos_token_id
                )
            
            result = processor.decode(outputs[0], skip_special_tokens=True).strip().upper()
            
            if "CAR_DAMAGE" in result:
                return "car_damage", "Image classified as vehicle damage."
            elif "INJURY" in result:
                return "injury", "Image classified as personal injury."
            else:
                return "other", "This image does not show car damage or personal injury."
            
        except Exception as e:
            logger.error(f"Classification error: {e}")
            return "other", f"Error: {str(e)}"

# ============================================================================
# CREATE GLOBAL INSTANCE
# ============================================================================

vision_processor = VisionProcessor()
print("✅ Vision Processor ready!")

model_manager._vision_model = vision_processor
model_manager._vision_processor = vision_processor
model_manager._vision_classes = ["car_damage", "injury", "other"]

print("✅ Vision model stored in Model Manager!")
print(f"📊 Model: Qwen2.5-VL-3B")
print(f"📊 Device: {vision_device}")
print(f"📊 Available: {vision_processor.available}")

In [ ]:
# ============================================================================
# GET DEEPSEEK API KEY FROM KAGGLE SECRETS
# ============================================================================

from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    DEEPSEEK_API_KEY = user_secrets.get_secret("DEEPSEEK_API_KEY")
    print("✅ DeepSeek API key loaded from Kaggle Secrets!")
except Exception as e:
    print(f"⚠️ Could not load DeepSeek API key from secrets: {e}")
    DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
    if DEEPSEEK_API_KEY:
        print("✅ DeepSeek API key loaded from environment variables!")
    else:
        print("❌ DeepSeek API key not found!")

# ============================================================================
# VISION ENDPOINTS (Using Qwen2.5-VL with Robust Error Handling)
# ============================================================================

import logging
logger = logging.getLogger(__name__)
import aiohttp
import os
import traceback

def process_image(file: UploadFile) -> Image.Image:
    """Read and process uploaded image with proper RGB conversion"""
    contents = file.file.read()
    image = Image.open(io.BytesIO(contents))
    
    if image.mode in ('RGBA', 'LA', 'P'):
        rgb_img = Image.new('RGB', image.size, (255, 255, 255))
        rgb_img.paste(image, mask=image.split()[-1] if image.mode == 'RGBA' else None)
        image = rgb_img
    elif image.mode != 'RGB':
        image = image.convert('RGB')
    
    return image

# ============================================================================
# DEEPSEEK HELPER FUNCTIONS
# ============================================================================

async def _call_deepseek(prompt: str) -> str:
    """Call DeepSeek API for insurance analysis"""
    try:
        api_key = DEEPSEEK_API_KEY  # ✅ Use the key from Secrets
        if not api_key:
            return "DeepSeek API key not configured. Please contact your insurance provider directly."
        
        async with aiohttp.ClientSession() as session:
            async with session.post(
                "https://api.deepseek.com/v1/chat/completions",
                headers={
                    "Authorization": f"Bearer {api_key}",
                    "Content-Type": "application/json"
                },
                json={
                    "model": "deepseek-chat",
                    "messages": [
                        {"role": "system", "content": "You are an expert insurance claims adjuster with extensive knowledge of auto and health insurance policies. Provide detailed, professional, and helpful responses."},
                        {"role": "user", "content": prompt}
                    ],
                    "temperature": 0.7,
                    "max_tokens": 500
                },
                timeout=aiohttp.ClientTimeout(total=30)
            ) as response:
                if response.status == 200:
                    data = await response.json()
                    return data['choices'][0]['message']['content']
                else:
                    return f"DeepSeek API error: {response.status}. Please try again later."
    except Exception as e:
        logger.error(f"DeepSeek error: {e}")
        return "Unable to generate detailed analysis at this time. Please contact your insurance provider directly."

async def _get_image_description(image: Image.Image) -> str:
    """Get a description of the image using Qwen2.5-VL"""
    try:
        prompt = "Describe what you see in this image in 2-3 sentences. Focus on the main objects, any visible damage or injuries, and the overall scene. Be specific about what you observe."
        
        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt}
                ]
            }
        ]
        
        text_prompt = processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
        inputs = processor(
            text=[text_prompt],
            images=[image],
            padding=True,
            return_tensors="pt"
        )
        inputs = {k: v.to(vision_processor.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=50,
                do_sample=True,
                temperature=0.7,
                pad_token_id=processor.tokenizer.eos_token_id
            )
        
        return processor.decode(outputs[0], skip_special_tokens=True).strip()
    except Exception as e:
        logger.error(f"Image description error: {e}")
        return "Vehicle with visible damage"

# ============================================================================
# VISION ENDPOINTS
# ============================================================================

@app.post("/api/vision/classify")
async def classify_image_endpoint(file: UploadFile = File(...)):
    """Classify an uploaded image using Qwen2.5-VL with robust error handling"""
    try:
        # ✅ Validate file
        if not file.content_type.startswith('image/'):
            return JSONResponse(
                status_code=400, 
                content={"error": "File must be an image", "success": False}
            )
        
        try:
            image = process_image(file)
        except Exception as e:
            logger.error(f"Image processing failed: {e}")
            return JSONResponse(
                status_code=400,
                content={"error": f"Could not process image: {str(e)}", "success": False}
            )
        
        # ✅ Check if vision processor is available
        if not hasattr(vision_processor, 'identify_image_type'):
            return JSONResponse(
                status_code=503,
                content={"error": "Vision model not available", "success": False}
            )
        
        # ✅ Classify with timeout
        try:
            classification, message = vision_processor.identify_image_type(image)
        except Exception as e:
            logger.error(f"Classification failed: {e}")
            return JSONResponse(
                status_code=500,
                content={"error": f"Classification failed: {str(e)}", "success": False}
            )
        
        return {
            "success": True,
            "classification": classification,
            "message": message,
            "image_type": file.content_type,
        }
        
    except Exception as e:
        logger.error(f"Unexpected error: {e}")
        logger.error(traceback.format_exc())
        return JSONResponse(
            status_code=500,
            content={"error": f"An unexpected error occurred: {str(e)}", "success": False}
        )

@app.post("/api/vision/analyze")
async def analyze_image_endpoint(file: UploadFile = File(...)):
    """Analyze image with insurance recommendations using Qwen2.5-VL + DeepSeek"""
    try:
        if not file.content_type.startswith('image/'):
            return JSONResponse(400, {"error": "File must be an image", "success": False})
        
        try:
            image = process_image(file)
        except Exception as e:
            return JSONResponse(400, {"error": f"Could not process image: {str(e)}", "success": False})
        
        if not hasattr(vision_processor, 'identify_image_type'):
            return JSONResponse(503, {"error": "Vision model not available", "success": False})
        
        try:
            classification, message = vision_processor.identify_image_type(image)
        except Exception as e:
            return JSONResponse(500, {"error": f"Classification failed: {str(e)}", "success": False})
        
        # ✅ If it's car damage or injury, get detailed analysis from DeepSeek
        if classification in ["car_damage", "injury"]:
            try:
                # Get image description
                description = await _get_image_description(image)
                
                # Build DeepSeek prompt
                if classification == "car_damage":
                    deepseek_prompt = f"""You are an expert auto insurance adjuster. A customer has uploaded a photo of vehicle damage.

Image Description: {description}

Based on this image analysis, provide a detailed auto insurance assessment including:
1. What type of coverage applies (collision/comprehensive)
2. Estimated deductible information
3. Step-by-step claim filing instructions
4. What documentation they need to prepare
5. Any important exclusions or conditions

Provide a professional, helpful response with clear sections."""
                else:  # injury
                    deepseek_prompt = f"""You are an expert health insurance adjuster. A customer has uploaded a photo of an injury.

Image Description: {description}

Based on this image analysis, provide a detailed health insurance assessment including:
1. What type of coverage applies (medical/PIP/MedPay)
2. Estimated deductible and copay information
3. Step-by-step claim filing instructions
4. What documentation they need to prepare
5. Any important exclusions or conditions

Provide a professional, helpful response with clear sections."""
                
                # Get DeepSeek analysis
                deepseek_analysis = await _call_deepseek(deepseek_prompt)
                
                # ✅ Use DeepSeek analysis
                if classification == "car_damage":
                    recommendation = "🚗 Auto insurance claim recommended. Document damage with multiple photos and contact your insurer."
                    confidence = 0.85
                else:
                    recommendation = "🏥 Health insurance claim recommended. Seek medical attention immediately and document injuries."
                    confidence = 0.85
                
                return {
                    "success": True,
                    "classification": classification,
                    "confidence": confidence,
                    "recommendation": recommendation,
                    "message": message,
                    "analysis": deepseek_analysis,
                    "image_description": description
                }
            except Exception as e:
                logger.error(f"DeepSeek analysis failed: {e}")
                # ✅ Fallback to static response if DeepSeek fails
                if classification == "car_damage":
                    recommendation = "🚗 Auto insurance claim recommended. Document damage with multiple photos and contact your insurer."
                    confidence = 0.85
                    analysis = f"Image classified as car_damage with 85.0% confidence. {recommendation}"
                else:
                    recommendation = "🏥 Health insurance claim recommended. Seek medical attention immediately and document injuries."
                    confidence = 0.85
                    analysis = f"Image classified as injury with 85.0% confidence. {recommendation}"
                
                return {
                    "success": True,
                    "classification": classification,
                    "confidence": confidence,
                    "recommendation": recommendation,
                    "message": message,
                    "analysis": analysis
                }
        
        else:
            # ✅ Other classification - no insurance needed
            return {
                "success": True,
                "classification": classification,
                "confidence": 0.0,
                "recommendation": "No insurance claim needed based on image content.",
                "message": message,
                "analysis": "No insurance claim needed based on image content. Please upload a photo of car damage or injury for detailed analysis."
            }
        
    except Exception as e:
        logger.error(f"Unexpected error: {e}")
        return JSONResponse(500, {"error": f"An unexpected error occurred: {str(e)}", "success": False})

@app.get("/api/vision/status")
async def vision_status():
    """Check if vision model is loaded"""
    try:
        return {
            "loaded": getattr(vision_processor, 'available', False),
            "model": "Qwen2.5-VL-3B",
            "device": str(getattr(vision_processor, 'device', 'unknown')),
            "available": getattr(vision_processor, 'available', False)
        }
    except Exception as e:
        return {
            "loaded": False,
            "model": "Qwen2.5-VL-3B",
            "device": "error",
            "available": False,
            "error": str(e)
        }

print("\n✅ Vision endpoints added!")
print("   - POST /api/vision/classify  (classify image)")
print("   - POST /api/vision/analyze   (analyze with recommendations)")
print("   - GET /api/vision/status     (check status)")

In [ ]:
# ============================================================================
# START THE SERVER - RUN THIS LAST!
# ============================================================================

import os
import time
import threading
import nest_asyncio
from pyngrok import ngrok

nest_asyncio.apply()

print("🚀 Starting server setup...")

# Kill old ngrok
try:
    ngrok.kill()
    print("✅ Killed old ngrok")
except:
    pass

# Start ngrok
public_url = ngrok.connect(8080)
ngrok_url = public_url.public_url
print(f"✅ ngrok tunnel established!")
print(f"🔗 Public URL: {ngrok_url}")

# Store for later use
os.environ['KAGGLE_API_URL'] = ngrok_url
print(f"📝 KAGGLE_API_URL set to: {ngrok_url}")

# ============================================================================
# START FASTAPI SERVER
# ============================================================================

import uvicorn

def start_server():
    """Start the FastAPI server"""
    print("🚀 Starting FastAPI server on port 8080...")
    try:
        uvicorn.run(
            app,  # ✅ app is now defined (from Cell 10)
            host="0.0.0.0",
            port=8080,
            log_level="info"
        )
    except Exception as e:
        print(f"❌ Server error: {e}")

# Start server in background thread
server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

# Wait for server to start
time.sleep(3)

print("\n" + "=" * 70)
print("✅ SERVER IS RUNNING!")
print("=" * 70)
print(f"📍 API URL: {ngrok_url}")
print(f"📖 Health check: {ngrok_url}/health")
print(f"📋 Vision classify: {ngrok_url}/api/vision/classify")
print(f"📋 Vision analyze: {ngrok_url}/api/vision/analyze")
print(f"📋 RAG query: {ngrok_url}/api/rag/query")
print(f"📋 QLoRA query: {ngrok_url}/api/qlora/query")
print("=" * 70)

# Test the health endpoint
try:
    import requests
    response = requests.get(f"{ngrok_url}/health", timeout=5)
    print(f"\n✅ Health check response: {response.status_code}")
    print(f"   {response.json()}")
except Exception as e:
    print(f"\n⚠️ Health check failed: {e}")
    print("   Waiting a few more seconds for server to fully start...")
    time.sleep(5)
    try:
        response = requests.get(f"{ngrok_url}/health", timeout=5)
        print(f"✅ Health check response: {response.status_code}")
        print(f"   {response.json()}")
    except:
        print("❌ Server still not responding. Check the cell output above for errors.")